# Decoding Functional Drift — SLURM Orchestrator

Tests whether XGBoost semantic-category classifiers trained on one day generalize to
future days, with permutation testing for significance.

**Phase 1** (one SLURM job per (patient, train_date, resample_idx)):  
Fit a full-day XGBoost model using the same class-balanced resample (same seed) as the
existing per-day results, trained on ALL class-balanced words (no 80/20 holdout), with
fixed hyperparameters from the existing best_params JSON.

**Phase 2** (one SLURM job per (patient, train_date, resample_idx), depends on Phase 1):  
For each subsequent test_date, apply the Phase-1 model (standardized with training mu/sd)
to test-day words and run N permutation tests (shuffle training X, refit mu/sd + XGBoost,
evaluate) in parallel via joblib.

Set `SPEECH_TYPE` to `'filtered'` or `'patient'`.

In [19]:
from pathlib import Path
import subprocess

import dill as pickle
import numpy as np
import pandas as pd

In [20]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PROJ_DIR  = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT  = PROJ_DIR / 'vad_new'

WORKER_TRAIN = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                    '/functional_drift/decoding_drift_train_worker.py')
WORKER_TEST  = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                    '/functional_drift/decoding_drift_test_worker.py')
PYTHON_BIN   = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS       = None   # None => scan all Y* folders
FORCE_RESUBMIT = False

# ── Speech type ────────────────────────────────────────────────────────────────
SPEECH_TYPE = 'filtered'   # 'filtered' or 'patient'

DEC_SOURCE_RUN = {
    'filtered': 'scat_xgboost_sampled_norm_filtered_per_day',
    'patient':  'scat_xgboost_sampled_norm_patient_per_day',
}[SPEECH_TYPE]

# Encoding source run that holds word_idx .npy files (used for test-day word indices)
ENC_SOURCE_RUN = {
    'filtered': 'word_level_duration_cv_filtered_speech_per_day',
    'patient':  'word_level_duration_cv_patient_speech_per_day',
}[SPEECH_TYPE]

DRIFT_SUBDIR = f'decoding_{SPEECH_TYPE}'

# ── Permutation testing ────────────────────────────────────────────────────────
N_PERMUTATIONS = 50
N_JOBS_PERM    = 4
SEED_STRIDE    = 42

# ── Valid semantic categories ──────────────────────────────────────────────────
VALID_CAT_MIN = 0
VALID_CAT_MAX = 9   # 10 = 'other', excluded

# ── SLURM — Phase 1 (CPU, XGBoost fit with GPU if available) ──────────────────
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'

P1_PARTITION = 'Universal'
P1_GRES      = 'gpu:1'
P1_CPUS      = 8
P1_MEM       = '32G'
P1_TIME      = '04:00:00'

# ── SLURM — Phase 2 (CPU, permutation testing) ────────────────────────────────
P2_PARTITION = 'Universal'
P2_GRES      = ''
P2_CPUS      = N_JOBS_PERM + 2
P2_MEM       = '32G'
P2_TIME      = '24:00:00'

LOGS_DIR    = VAD_ROOT / f'decoding_drift_{SPEECH_TYPE}_slurm_logs'
SCRIPTS_DIR = VAD_ROOT / f'decoding_drift_{SPEECH_TYPE}_slurm_scripts'
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:       ', VAD_ROOT)
print('dec source run: ', DEC_SOURCE_RUN)
print('enc source run: ', ENC_SOURCE_RUN)
print('drift dir:      ', DRIFT_SUBDIR)
print(f'permutations: {N_PERMUTATIONS} x n_jobs={N_JOBS_PERM}')

vad root:        /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
dec source run:  scat_xgboost_sampled_norm_filtered_per_day
enc source run:  word_level_duration_cv_filtered_speech_per_day
drift dir:       decoding_filtered
permutations: 50 x n_jobs=4


## 1. Discover patients, training days, and resamples

In [21]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def resolve_patient_neural_inputs(patient):
    r = VAD_ROOT / patient
    frs = first_existing([
        r / 'neural_embeddings' / 'word_frs.npy',
        r / 'all_convo_recording' / 'word_avg_frs.npy',
    ])
    cats = first_existing([
        r / 'embeddings' / 'semantic_cluster_predictions.npy',
    ])
    return frs, cats


def get_model_days_dec(patient):
    """Days with existing per-day decoding results (have at least one best_params JSON)."""
    base = VAD_ROOT / patient / 'standard_decoding_analysis' / DEC_SOURCE_RUN
    if not base.exists():
        return []
    days = []
    for d in sorted(base.iterdir()):
        if not d.is_dir():
            continue
        params_files = list(d.glob('best_params_resample_*.json'))
        idx_files    = list(d.glob(f'{patient}_*_word_idx.npy'))
        if params_files and idx_files:
            resample_ids = sorted(
                int(p.stem.split('_')[-1]) for p in params_files
            )
            days.append((d.name, idx_files[0], resample_ids))
    return days


def get_test_days_enc(patient):
    """Days with word_idx from the ENCODING source run (used as test-day data)."""
    base = VAD_ROOT / patient / 'encoding' / ENC_SOURCE_RUN
    if not base.exists():
        return []
    return [d.name for d in sorted(base.iterdir())
            if d.is_dir() and list(d.glob(f'{patient}_*_word_idx.npy'))]


if PATIENTS is None:
    all_patients = sorted(p.name for p in VAD_ROOT.iterdir()
                          if p.is_dir() and p.name.startswith('Y'))
else:
    all_patients = list(PATIENTS)

overview = []
for pt in all_patients:
    mdays = get_model_days_dec(pt)
    tdays = get_test_days_enc(pt)
    n_pairs = sum(1 for (d, _, rs) in mdays for t in tdays if t > d) * (
        len(mdays[0][2]) if mdays else 0
    ) if mdays else 0
    overview.append(dict(patient=pt, n_model_days=len(mdays),
                         n_test_days=len(tdays),
                         n_resamples=len(mdays[0][2]) if mdays else 0,
                         n_p1_jobs=sum(len(rs) for _, _, rs in mdays)))

overview_df = pd.DataFrame(overview)
print(f'Patients with model days: {int((overview_df.n_model_days > 0).sum())}')
print(f'Total Phase-1 jobs needed: {int(overview_df.n_p1_jobs.sum())}')
display(overview_df)

Patients with model days: 20
Total Phase-1 jobs needed: 2900


,patient,n_model_days,n_test_days,n_resamples,n_p1_jobs
0,YEY,2,2,20,40
1,YEZ,6,6,20,120
2,YFA,9,9,20,180
3,YFB,9,9,20,180
4,YFC,9,9,20,180
5,YFD,6,6,20,120
6,YFE,4,4,20,80
7,YFF,10,10,20,200
8,YFG,4,5,20,80
9,YFI,7,7,20,140


## 2. Phase-1 Status and Submission

One job per (patient, train_date, resample_idx).

In [22]:
def p1_paths_dec(patient, train_date, r):
    drift_dir = VAD_ROOT / patient / 'functional_drift' / DRIFT_SUBDIR / train_date / f'r{r}'
    return dict(
        out_dir    = drift_dir,
        model_pkl  = drift_dir / f'{patient}_r{r}_fullday_model.pkl',
        meta_json  = drift_dir / f'{patient}_r{r}_meta.json',
        success    = drift_dir / f'r{r}_TRAIN_SUCCESS',
        error      = drift_dir / f'r{r}_error.txt',
    )


rows_p1 = []
for pt in all_patients:
    frs_path, cats_path = resolve_patient_neural_inputs(pt)
    if frs_path is None or cats_path is None:
        continue
    for train_date, word_idx_path, resample_ids in get_model_days_dec(pt):
        dec_day_dir = VAD_ROOT / pt / 'standard_decoding_analysis' / DEC_SOURCE_RUN / train_date
        for r in resample_ids:
            params_json = dec_day_dir / f'best_params_resample_{r}.json'
            if not params_json.exists():
                continue
            pp = p1_paths_dec(pt, train_date, r)
            done  = pp['success'].exists()
            error = pp['error'].exists()
            rows_p1.append(dict(
                patient=pt, train_date=train_date, resample_idx=r,
                best_params_json=params_json,
                word_idx_path=word_idx_path,
                frs_path=frs_path,
                cluster_preds_path=cats_path,
                **pp,
                done=done, has_error=error,
                needs_p1=not done or FORCE_RESUBMIT,
            ))

p1_df = pd.DataFrame(rows_p1)
print(f'Phase-1 rows: {len(p1_df)}')
print(f'  done:         {p1_df.done.sum()}')
print(f'  errors:       {p1_df.has_error.sum()}')
print(f'  needs submit: {p1_df.needs_p1.sum()}')
p1_df[['patient','train_date','resample_idx','done','has_error','needs_p1']].head(20)

Phase-1 rows: 2900
  done:         2860
  errors:       40
  needs submit: 40


,patient,train_date,resample_idx,done,has_error,needs_p1
0,YEY,2024-04-01,0,True,False,False
1,YEY,2024-04-01,1,True,False,False
2,YEY,2024-04-01,2,True,False,False
3,YEY,2024-04-01,3,True,False,False
4,YEY,2024-04-01,4,True,False,False
5,YEY,2024-04-01,5,True,False,False
6,YEY,2024-04-01,6,True,False,False
7,YEY,2024-04-01,7,True,False,False
8,YEY,2024-04-01,8,True,False,False
9,YEY,2024-04-01,9,True,False,False


In [23]:
DRY_RUN = False

submitted_p1 = {}   # (patient, train_date, r) -> job_id
failed_p1    = []

for _, row in p1_df[p1_df['needs_p1']].iterrows():
    pt         = row['patient']
    train_date = row['train_date']
    r          = int(row['resample_idx'])
    out_dir    = row['out_dir']
    out_dir.mkdir(parents=True, exist_ok=True)

    if FORCE_RESUBMIT:
        reset = f"rm -f {row['success']} {row['error']}"
    else:
        reset = 'true'

    gres_line = f'#SBATCH --gres={P1_GRES}' if P1_GRES else ''
    cmd = (
        f'{PYTHON_BIN} {WORKER_TRAIN} '
        f'--patient {pt} '
        f'--vad-root {VAD_ROOT} '
        f'--out-dir {out_dir} '
        f'--train-date {train_date} '
        f'--resample-idx {r} '
        f'--best-params-json {row["best_params_json"]} '
        f'--word-idx-path {row["word_idx_path"]} '
        f'--cluster-preds-path {row["cluster_preds_path"]} '
        f'--frs-path {row["frs_path"]} '
        f'--seed-stride {SEED_STRIDE} '
        f'--n-jobs {P1_CPUS}'
    )

    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=dec_drift_train_{pt}_{train_date}_r{r}',
        f'#SBATCH --partition={P1_PARTITION}',
        gres_line,
        f'#SBATCH --cpus-per-task={P1_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        #f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P1_MEM}',
        f'#SBATCH --time={P1_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_{train_date}_r{r}_p1_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_{train_date}_r{r}_p1_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"',
        reset,
        cmd,
        'echo "END: $(date)"',
    ])

    sbatch_path = SCRIPTS_DIR / f'{pt}_{train_date}_r{r}_p1.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] P1: {pt} {train_date} r={r}')
        continue

    try:
        res = subprocess.run(['sbatch', '--parsable', str(sbatch_path)],
                             capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p1[(pt, train_date, r)] = jid
        print(f'submitted P1: {pt} {train_date} r={r} -> {jid}')
    except subprocess.CalledProcessError as exc:
        failed_p1.append((pt, train_date, r, exc.stderr.strip()))
        print(f'FAILED P1: {pt} {train_date} r={r}\n{exc.stderr}')

print(f'\nP1 submitted={len(submitted_p1)}, failed={len(failed_p1)}')

submitted P1: YFC 2024-07-18 r=0 -> 485850
submitted P1: YFC 2024-07-18 r=1 -> 485851
submitted P1: YFC 2024-07-18 r=2 -> 485852
submitted P1: YFC 2024-07-18 r=3 -> 485853
submitted P1: YFC 2024-07-18 r=4 -> 485854
submitted P1: YFC 2024-07-18 r=5 -> 485855
submitted P1: YFC 2024-07-18 r=6 -> 485856
submitted P1: YFC 2024-07-18 r=7 -> 485857
submitted P1: YFC 2024-07-18 r=8 -> 485858
submitted P1: YFC 2024-07-18 r=9 -> 485859
submitted P1: YFC 2024-07-18 r=10 -> 485860
submitted P1: YFC 2024-07-18 r=11 -> 485861
submitted P1: YFC 2024-07-18 r=12 -> 485862
submitted P1: YFC 2024-07-18 r=13 -> 485863
submitted P1: YFC 2024-07-18 r=14 -> 485864
submitted P1: YFC 2024-07-18 r=15 -> 485865
submitted P1: YFC 2024-07-18 r=16 -> 485866
submitted P1: YFC 2024-07-18 r=17 -> 485867
submitted P1: YFC 2024-07-18 r=18 -> 485868
submitted P1: YFC 2024-07-18 r=19 -> 485869
submitted P1: YFK 2025-02-10 r=0 -> 485870
submitted P1: YFK 2025-02-10 r=1 -> 485871
submitted P1: YFK 2025-02-10 r=2 -> 485872
s

## 3. Phase-2 Status and Submission

One job per (patient, train_date, resample_idx) — tests all subsequent days with permutation testing.

In [24]:
def p2_success_dec(patient, train_date, r, test_date, resample_dir):
    return resample_dir / f'r{r}_{test_date}_DRIFT_SUCCESS'

def p2_result_dec(patient, train_date, r, test_date, resample_dir):
    return resample_dir / f'{patient}_{train_date}_{test_date}_r{r}_drift.pkl'


rows_p2 = []
for pt in all_patients:
    test_days = get_test_days_enc(pt)
    frs_path, cats_path = resolve_patient_neural_inputs(pt)
    if frs_path is None or cats_path is None:
        continue
    for train_date, word_idx_path, resample_ids in get_model_days_dec(pt):
        future_tests = [t for t in test_days if t > train_date]
        if not future_tests:
            continue
        for r in resample_ids:
            pp = p1_paths_dec(pt, train_date, r)
            all_test_done = all(
                p2_success_dec(pt, train_date, r, td, pp['out_dir']).exists()
                for td in future_tests
            )
            rows_p2.append(dict(
                patient=pt, train_date=train_date, resample_idx=r,
                resample_dir=pp['out_dir'],
                future_tests=future_tests,
                frs_path=frs_path,
                cluster_preds_path=cats_path,
                p1_done=pp['success'].exists(),
                p1_jid=submitted_p1.get((pt, train_date, r), None),
                all_test_done=all_test_done,
            ))

p2_df = pd.DataFrame(rows_p2)
print(f'Phase-2 rows: {len(p2_df)}')
print(f'  P1 done:        {p2_df.p1_done.sum()}')
print(f'  P1 pending:     {p2_df.p1_jid.notna().sum()}')
print(f'  All tests done: {p2_df.all_test_done.sum()}')

Phase-2 rows: 2500
  P1 done:        2460
  P1 pending:     40
  All tests done: 2460


In [25]:
DRY_RUN = False

submitted_p2 = {}
failed_p2    = []

for _, row in p2_df.iterrows():
    pt         = row['patient']
    train_date = row['train_date']
    r          = int(row['resample_idx'])

    if not row['p1_done'] and row['p1_jid'] is None:
        continue
    if row['all_test_done'] and not FORCE_RESUBMIT:
        continue

    resample_dir = row['resample_dir']
    resample_dir.mkdir(parents=True, exist_ok=True)

    dep_flag = ''
    if row['p1_jid']:
        dep_flag = f'--dependency=afterok:{row["p1_jid"]}'
    elif not row['p1_done']:
        print(f'  skipping {pt} {train_date} r={r}: P1 not done and no pending job ID')
        continue

    gres_line = f'#SBATCH --gres={P2_GRES}' if P2_GRES else ''
    cmd = (
        f'{PYTHON_BIN} {WORKER_TEST} '
        f'--patient {pt} '
        f'--vad-root {VAD_ROOT} '
        f'--train-date {train_date} '
        f'--resample-idx {r} '
        f'--train-out-dir {resample_dir} '
        f'--source-run {DEC_SOURCE_RUN} '
        f'--word-idx-source-run {ENC_SOURCE_RUN} '
        f'--cluster-preds-path {row["cluster_preds_path"]} '
        f'--frs-path {row["frs_path"]} '
        f'--n-permutations {N_PERMUTATIONS} '
        f'--n-jobs {N_JOBS_PERM} '
        f'--valid-cat-min {VALID_CAT_MIN} '
        f'--valid-cat-max {VALID_CAT_MAX}'
    )

    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=dec_drift_test_{pt}_{train_date}_r{r}',
        f'#SBATCH --partition={P2_PARTITION}',
        gres_line,
        f'#SBATCH --cpus-per-task={P2_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        #f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P2_MEM}',
        f'#SBATCH --time={P2_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_{train_date}_r{r}_p2_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_{train_date}_r{r}_p2_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"',
        cmd,
        'echo "END: $(date)"',
    ])

    sbatch_path = SCRIPTS_DIR / f'{pt}_{train_date}_r{r}_p2.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] P2: {pt} {train_date} r={r} dep={dep_flag}')
        continue

    try:
        sbatch_args = ['sbatch', '--parsable']
        if dep_flag:
            sbatch_args.append(dep_flag)
        sbatch_args.append(str(sbatch_path))
        res = subprocess.run(sbatch_args, capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p2[(pt, train_date, r)] = jid
        print(f'submitted P2: {pt} {train_date} r={r} -> {jid}  dep={dep_flag or "none"}')
    except subprocess.CalledProcessError as exc:
        failed_p2.append((pt, train_date, r, exc.stderr.strip()))
        print(f'FAILED P2: {pt} {train_date} r={r}\n{exc.stderr}')

print(f'\nP2 submitted={len(submitted_p2)}, failed={len(failed_p2)}')

submitted P2: YFC 2024-07-18 r=0 -> 485890  dep=--dependency=afterok:485850
submitted P2: YFC 2024-07-18 r=1 -> 485891  dep=--dependency=afterok:485851
submitted P2: YFC 2024-07-18 r=2 -> 485892  dep=--dependency=afterok:485852
submitted P2: YFC 2024-07-18 r=3 -> 485893  dep=--dependency=afterok:485853
submitted P2: YFC 2024-07-18 r=4 -> 485894  dep=--dependency=afterok:485854
submitted P2: YFC 2024-07-18 r=5 -> 485895  dep=--dependency=afterok:485855
submitted P2: YFC 2024-07-18 r=6 -> 485896  dep=--dependency=afterok:485856
submitted P2: YFC 2024-07-18 r=7 -> 485897  dep=--dependency=afterok:485857
submitted P2: YFC 2024-07-18 r=8 -> 485898  dep=--dependency=afterok:485858
submitted P2: YFC 2024-07-18 r=9 -> 485899  dep=--dependency=afterok:485859
submitted P2: YFC 2024-07-18 r=10 -> 485900  dep=--dependency=afterok:485860
submitted P2: YFC 2024-07-18 r=11 -> 485901  dep=--dependency=afterok:485861
submitted P2: YFC 2024-07-18 r=12 -> 485902  dep=--dependency=afterok:485862
submitted

## 4. Check Status

In [26]:
rows_status = []
for pt in all_patients:
    test_days = get_test_days_enc(pt)
    for train_date, _, resample_ids in get_model_days_dec(pt):
        for r in resample_ids:
            pp = p1_paths_dec(pt, train_date, r)
            for test_date in [t for t in test_days if t > train_date]:
                rows_status.append(dict(
                    patient=pt, train_date=train_date, resample_idx=r, test_date=test_date,
                    p1_done=pp['success'].exists(),
                    p2_done=p2_success_dec(pt, train_date, r, test_date, pp['out_dir']).exists(),
                    result_exists=p2_result_dec(pt, train_date, r, test_date, pp['out_dir']).exists(),
                    p2_error=(pp['out_dir'] / f'r{r}_{test_date}_error.txt').exists(),
                ))

status_df = pd.DataFrame(rows_status)
print(f'Total (train, resample, test) triples: {len(status_df)}')
print(f'  P1 done:    {status_df.p1_done.sum()}')
print(f'  P2 done:    {status_df.p2_done.sum()}')
print(f'  P2 errors:  {status_df.p2_error.sum()}')
status_df[['patient','train_date','resample_idx','test_date','p1_done','p2_done','result_exists','p2_error']].head(20)

Total (train, resample, test) triples: 10220
  P1 done:    9900
  P2 done:    9900
  P2 errors:  0


,patient,train_date,resample_idx,test_date,p1_done,p2_done,result_exists,p2_error
0,YEY,2024-04-01,0,2024-04-02,True,True,True,False
1,YEY,2024-04-01,1,2024-04-02,True,True,True,False
2,YEY,2024-04-01,2,2024-04-02,True,True,True,False
3,YEY,2024-04-01,3,2024-04-02,True,True,True,False
4,YEY,2024-04-01,4,2024-04-02,True,True,True,False
5,YEY,2024-04-01,5,2024-04-02,True,True,True,False
6,YEY,2024-04-01,6,2024-04-02,True,True,True,False
7,YEY,2024-04-01,7,2024-04-02,True,True,True,False
8,YEY,2024-04-01,8,2024-04-02,True,True,True,False
9,YEY,2024-04-01,9,2024-04-02,True,True,True,False


## 5. Assemble Results

In [27]:
all_results = []

for pt in all_patients:
    test_days = get_test_days_enc(pt)
    for train_date, _, resample_ids in get_model_days_dec(pt):
        for r in resample_ids:
            pp = p1_paths_dec(pt, train_date, r)
            resample_dir = pp['out_dir']
            for test_date in [t for t in test_days if t > train_date]:
                result_path = p2_result_dec(pt, train_date, r, test_date, resample_dir)
                if not result_path.exists():
                    continue
                with open(result_path, 'rb') as f:
                    res = pickle.load(f)
                all_results.append(dict(
                    patient=res['patient'],
                    train_date=res['train_date'],
                    test_date=res['test_date'],
                    resample_idx=res['resample_idx'],
                    day_offset=res['day_offset'],
                    n_train_words=res['n_train_words'],
                    n_test_words=res['n_test_words'],
                    n_classes_train=res['n_classes_train'],
                    n_classes_test=res['n_classes_test'],
                    n_permutations=res['n_permutations'],
                    acc=res['acc'],
                    balanced_accuracy=res['balanced_accuracy'],
                    f1_macro=res['f1_macro'],
                    auc_macro_ovr=res['auc_macro_ovr'],
                    perm_acc_mean=float(res['perm_acc'].mean()),
                    perm_bacc_mean=float(res['perm_balanced_accuracy'].mean()),
                    perm_f1_mean=float(res['perm_f1_macro'].mean()),
                    p_val_acc=res['p_val_acc'],
                    p_val_bacc=res['p_val_bacc'],
                    p_val_f1=res['p_val_f1'],
                    p_val_auc=res['p_val_auc'],
                ))

if all_results:
    combined = pd.DataFrame(all_results)
    out_path = VAD_ROOT / f'functional_drift_decoding_{SPEECH_TYPE}_results_all.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump(combined, f)
    print(f'combined: {len(combined)} rows -> {out_path}')
    display(combined.head())
else:
    print('No results assembled yet.')

combined: 9900 rows -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/functional_drift_decoding_filtered_results_all.pkl


,patient,train_date,test_date,resample_idx,day_offset,n_train_words,n_test_words,n_classes_train,n_classes_test,n_permutations,...,balanced_accuracy,f1_macro,auc_macro_ovr,perm_acc_mean,perm_bacc_mean,perm_f1_mean,p_val_acc,p_val_bacc,p_val_f1,p_val_auc
0,YEY,2024-04-01,2024-04-02,0,1,3871,10909,10,10,50,...,0.110827,0.100011,0.512905,0.142030,0.099613,0.088591,0.019608,0.019608,0.019608,0.019608
1,YEY,2024-04-01,2024-04-02,1,1,3871,10909,10,10,50,...,0.118363,0.109383,0.535656,0.154318,0.100333,0.087450,0.019608,0.019608,0.019608,0.019608
2,YEY,2024-04-01,2024-04-02,2,1,3871,10909,10,10,50,...,0.113922,0.105259,0.521056,0.144506,0.099235,0.087441,0.019608,0.019608,0.019608,0.019608
3,YEY,2024-04-01,2024-04-02,3,1,3871,10909,10,10,50,...,0.115009,0.106990,0.528612,0.146155,0.098917,0.087448,0.019608,0.019608,0.019608,0.019608
4,YEY,2024-04-01,2024-04-02,4,1,3871,10909,10,10,50,...,0.115742,0.107726,0.528118,0.147726,0.099190,0.087973,0.019608,0.019608,0.019608,0.019608


## 5b. Add f1_micro to Combined Results

For standard multiclass classification, **f1_micro == accuracy** (micro-averaging pools
TP/FP/FN across all classes, which reduces to total-correct / total-predictions).
We can therefore derive it from the already-stored `acc` column without re-running any jobs.

This cell loads the saved pickle (if not already in memory), adds `f1_micro`,
`perm_f1_micro_mean`, and `p_val_f1_micro`, then re-saves.

In [28]:
out_path = VAD_ROOT / f'functional_drift_decoding_{SPEECH_TYPE}_results_all.pkl'

# Load from disk if combined isn't already in memory from the assemble cell
if 'combined' not in dir() or combined is None or len(combined) == 0:
    with open(out_path, 'rb') as f:
        combined = pickle.load(f)
    print(f'loaded {len(combined)} rows from {out_path}')

# f1_micro == accuracy for standard multiclass classification
combined['f1_micro'] = combined['acc']
combined['perm_f1_micro_mean'] = combined['perm_acc_mean']
combined['p_val_f1_micro'] = combined['p_val_acc']

with open(out_path, 'wb') as f:
    pickle.dump(combined, f)

print(f'saved {len(combined)} rows with f1_micro -> {out_path}')
print(f'f1_micro range: [{combined.f1_micro.min():.4f}, {combined.f1_micro.max():.4f}]')
print(f'columns now: {list(combined.columns)}')

saved 9900 rows with f1_micro -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/functional_drift_decoding_filtered_results_all.pkl
f1_micro range: [0.0660, 0.3431]
columns now: ['patient', 'train_date', 'test_date', 'resample_idx', 'day_offset', 'n_train_words', 'n_test_words', 'n_classes_train', 'n_classes_test', 'n_permutations', 'acc', 'balanced_accuracy', 'f1_macro', 'auc_macro_ovr', 'perm_acc_mean', 'perm_bacc_mean', 'perm_f1_mean', 'p_val_acc', 'p_val_bacc', 'p_val_f1', 'p_val_auc', 'f1_micro', 'perm_f1_micro_mean', 'p_val_f1_micro']


## 6. Summary Statistics

In [29]:
if all_results:
    combined = pd.DataFrame(all_results) if not isinstance(all_results, pd.DataFrame) else combined

    # Average across resamples per (patient, train_date, test_date)
    resample_avg = (
        combined.groupby(['patient','train_date','test_date','day_offset'])
        .agg(
            acc_mean=('acc','mean'), acc_std=('acc','std'),
            balanced_accuracy_mean=('balanced_accuracy','mean'),
            f1_macro_mean=('f1_macro','mean'),
            auc_mean=('auc_macro_ovr','mean'),
            p_val_f1_mean=('p_val_f1','mean'),
            frac_sig_05=('p_val_f1', lambda x: (x < 0.05).mean()),
            n_resamples=('resample_idx','nunique'),
        )
        .reset_index()
    )

    by_offset = (
        resample_avg.groupby('day_offset')
        .agg(
            n_pairs=('patient','count'),
            acc_mean=('acc_mean','mean'),
            balanced_accuracy_mean=('balanced_accuracy_mean','mean'),
            f1_macro_mean=('f1_macro_mean','mean'),
            frac_sig_05_mean=('frac_sig_05','mean'),
        )
        .reset_index()
    )
    print('By day offset:')
    display(by_offset)

By day offset:


,day_offset,n_pairs,acc_mean,balanced_accuracy_mean,f1_macro_mean,frac_sig_05_mean
0,1,121,0.229019,0.126388,0.113278,0.902479
1,2,101,0.231423,0.123171,0.107801,0.824257
2,3,83,0.231607,0.120450,0.103850,0.745181
3,4,65,0.233779,0.119786,0.101260,0.690769
4,5,50,0.231478,0.118261,0.098313,0.655000
5,6,34,0.227988,0.117889,0.097615,0.616176
6,7,21,0.213286,0.115808,0.095628,0.602381
7,8,12,0.208017,0.114693,0.093814,0.570833
8,9,6,0.200186,0.111939,0.088217,0.483333
9,10,2,0.201170,0.114321,0.094925,0.475000
